In [6]:
!rm -rf predictive-maintenance-cmapss
!git clone https://github.com/MaissaBM2003/predictive-maintenance-cmapss.git
%cd predictive-maintenance-cmapss
!pip install -q mlflow pandera lightgbm

Cloning into 'predictive-maintenance-cmapss'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 81 (delta 24), reused 72 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 1.21 MiB | 2.82 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/kaggle/working/predictive-maintenance-cmapss/predictive-maintenance-cmapss


In [7]:
!mkdir -p data
!ln -s /kaggle/input/datasets/maissabenmrad/nasa-cmaps/CMaps data/raw

In [8]:
!python -m src.preprocessing

/usr/local/lib/python3.12/dist-packages/pandera/_pandas_deprecated.py:144: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)
Train : (14241, 30, 14) | Val : (3490, 30, 14) | Test : (100, 30, 14)


In [9]:
!python -m src.train_baseline

linear_regression  | CV RMSE 17.75 ± 0.28 | Val RMSE 15.06 | Test RMSE 15.71
random_forest      | CV RMSE 15.34 ± 1.05 | Val RMSE 12.96 | Test RMSE 13.14
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X d

In [10]:
import mlflow, mlflow.lightgbm
import numpy as np
import matplotlib.pyplot as plt
from src.features import window_features

mlflow.set_experiment("predictive-maintenance")
runs = mlflow.search_runs(filter_string="tags.mlflow.runName = 'baseline_lightgbm'",
                          order_by=["start_time DESC"])
model = mlflow.lightgbm.load_model(f"runs:/{runs.iloc[0].run_id}/model")

d = np.load("data/processed/windows.npz")
X_test, y_test = window_features(d["X_test"]), d["y_test"]
p_test = np.clip(model.predict(X_test), 0, None)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, p_test, alpha=0.7)
plt.plot([0, 125], [0, 125], "r--")
plt.xlabel("Vraie RUL")
plt.ylabel("RUL prédite")
plt.title("LightGBM : prédictions sur le test FD001")
plt.show()

ImportError: cannot import name 'EvaluationDataset' from 'mlflow.entities' (/usr/local/lib/python3.12/dist-packages/mlflow/entities/__init__.py)

In [ ]:
import mlflow
mlflow.set_experiment("predictive-maintenance")

runs = mlflow.search_runs(order_by=["metrics.cv_rmse_mean ASC"])
cols = ["tags.mlflow.runName", "metrics.cv_rmse_mean", "metrics.cv_rmse_std",
        "metrics.val_rmse", "metrics.test_rmse", "metrics.test_mae", "metrics.test_nasa_score"]
runs[cols]

In [ ]:
!ls
!zip -rq /kaggle/working/mlflow_results.zip mlflow.db mlruns

In [ ]:
!pwd
!ls


In [11]:
!cp /kaggle/working/predictive-maintenance-cmapss/mlflow.db /kaggle/working/mlflow_kaggle.db
!ls -la /kaggle/working

total 900
drwxr-xr-x  4 root root   4096 Oct  1 13:51 .
drwxr-xr-x  5 root root   4096 Oct  1 12:55 ..
-rw-r--r--  1 root root 876544 Oct  1 13:51 mlflow_kaggle.db
-rw-r--r--  1 root root  28341 Oct  1 13:10 mlflow_results.zip
drwxr-xr-x 10 root root   4096 Oct  1 13:32 predictive-maintenance-cmapss
drwxr-xr-x  2 root root   4096 Oct  1 12:56 .virtual_documents


In [12]:
  !cp mlflow.db /kaggle/working/mlflow_kaggle.db